In [1]:
import json
from pathlib import Path
from pprint import pprint

## I. functions used 

In [2]:
TYPE_RENAME = {
    "DistractorNegation": "distractor_negation",
    "DistractorLexicalEdit": "distractor_lexical",
    "DistractorAssociative0": "distractor_associative",
}



def open_and_show_json(path: Path):
    """ opens the JSON file with data and prints out basic info about it."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"loaded items: {len(data)}")
    print("\nfirst item keys:")
    print(list(data[0].keys()))

    print("\nexample options:")
    pprint(data[0]["options"])

    print("\nexample item meta_general:")
    pprint(data[0]["meta_general"])

    print("\nexample item meta_special:")
    pprint(data[0]["meta_special"])
    return data





def convert_options(old_options: dict) -> tuple[list[dict], str]:
    """ converts the old dict with answer options while renaming distractors from camelcase + deriving correct option"""
    labels = ["A", "B", "C", "D"]

    order = [
        "presupposition",
        "DistractorNegation",
        "DistractorLexicalEdit",
        "DistractorAssociative0",
    ]

    converted = []
    for label, key in zip(labels, [k for k in order if k in old_options]):
        if key == "presupposition":
            new_type = "presupposition"
        else:
            new_type = TYPE_RENAME[key]

        converted.append({
            "label": label,
            "type": new_type,
            "text": old_options[key],
        })

    # we presuppose that the correct option is the one with type 'presupposition'
    correct_option = next(opt["label"] for opt in converted if opt["type"] == "presupposition")
    return converted, correct_option


def convert_item(item: dict) -> dict:
    """Converts the old presupposition item to the new standardized schema."""
    new_options, correct_option = convert_options(item["options"])

    meta_general = item.get("meta_general", {}) or {}
    meta_special = item.get("meta_special", {}) or {}

    creation_method = (
        meta_general.get("creation_method")
        or meta_general.get("extraction_method")
        or meta_general.get("source")
        or ""
    )

    # optional normalization, if you want the new label
    if creation_method == "cql_query":
        creation_method = "corpus_mining"

    metadata = {
        "corpus": meta_general.get("corpus", ""),
        "genre": meta_general.get("genre", ""),
        "trigger": meta_special.get("trigger", item.get("category", "")),
        "negation": meta_special.get("negation", "")
    }

    # remove empty metadata fields
    metadata = {k: v for k, v in metadata.items() if v not in ("", None, [], {})}

    return {
        "id": item["id"],
        "creation_method": creation_method,
        "phenomenon": item["phenomenon"],
        "category": item["category"],
        "context": item["context"],
        "utterance": item["utterance"],
        "question": item["question"],
        "options": new_options,
        "correct_option": correct_option,
        "metadata": metadata
    }


## II. notebook usage

In [20]:
# path to the presupposition json file
path_initial = Path(r"data\presupposition\initial")
file_name = "presupposition-factive-semifactive-60.json"
file_path = path_initial / file_name


# change-of-state: presupposition-change-of-state-60.json
# wh-question: presupposition-wh-question-60.json
# iterative: presupposition-iterative-60.json

In [21]:
# open the json with data
data = open_and_show_json(file_path)

loaded items: 60

first item keys:
['id', 'phenomenon', 'category', 'context', 'utterance', 'question', 'options', 'meta_general', 'meta_special']

example options:
{'DistractorAssociative0': 'Seaxmund´s Ham je v této oblasti největším '
                           'obchodním centrem.',
 'DistractorLexicalEdit': 'Mluvčí kdysi v Seaxmund´s Ham přenocoval při cestě '
                          'na jih.',
 'DistractorNegation': 'Mluvčí nikdy ze Seaxmund´s Ham nepocházel.',
 'presupposition_paraphrased': 'Mluvčí je původem ze Seaxmund´s Ham.',
 'presupposition_true': 'Mluvčí pochází ze Seaxmund´s Ham.'}

example item meta_general:
{'corpus': 'SYN2025',
 'extraction_method': 'cql_query',
 'genre': 'beletrie',
 'note': ''}

example item meta_special:
{'trigger': 'zapomenout'}


In [22]:
# pick an old item 
data[0]

{'id': 'presupposition-factive-semifactive-1',
 'phenomenon': 'presupposition',
 'category': 'factive-semifactive',
 'context': 'V napjatém rozhovoru si jeden z mužů ověřuje, zda si druhý pamatuje důležitou informaci, kterou mu už dříve říkal.',
 'utterance': 'Mluvčí: „A ty také, gerefo. Zapomněl jsi, že i já pocházím ze Seaxmund´s Ham?“',
 'question': 'Co je bráno jako samozřejmý předpoklad, aby výpověď dávala smysl?',
 'options': {'presupposition_true': 'Mluvčí pochází ze Seaxmund´s Ham.',
  'presupposition_paraphrased': 'Mluvčí je původem ze Seaxmund´s Ham.',
  'DistractorNegation': 'Mluvčí nikdy ze Seaxmund´s Ham nepocházel.',
  'DistractorLexicalEdit': 'Mluvčí kdysi v Seaxmund´s Ham přenocoval při cestě na jih.',
  'DistractorAssociative0': 'Seaxmund´s Ham je v této oblasti největším obchodním centrem.'},
 'meta_general': {'corpus': 'SYN2025',
  'genre': 'beletrie',
  'extraction_method': 'cql_query',
  'note': ''},
 'meta_special': {'trigger': 'zapomenout'}}

In [23]:
# CREATE A SINGLE RIGHT PRESUPPOSITION FIELD
# we will keep only one field with presupposition in the options, so either rename presupposition_true or presupposition_paraphrased


# presupposition_true so far only for possessives, comparatives;
# for others we take presupposition_paraphrased



field_to_take = "presupposition_paraphrased" # presupposition_true
field_to_delete = "presupposition_true"

for item in data:
    if field_to_take in item["options"]:
        item["options"]["presupposition"] = item["options"].pop(field_to_take)
    if field_to_delete in item["options"]:
        del item["options"][field_to_delete]

pprint(data[0]["options"])

{'DistractorAssociative0': 'Seaxmund´s Ham je v této oblasti největším '
                           'obchodním centrem.',
 'DistractorLexicalEdit': 'Mluvčí kdysi v Seaxmund´s Ham přenocoval při cestě '
                          'na jih.',
 'DistractorNegation': 'Mluvčí nikdy ze Seaxmund´s Ham nepocházel.',
 'presupposition': 'Mluvčí je původem ze Seaxmund´s Ham.'}


In [24]:
# look how new options will look
opts, correct = convert_options(data[0]["options"])
print(correct)
for x in opts:
    print(x)

A
{'label': 'A', 'type': 'presupposition', 'text': 'Mluvčí je původem ze Seaxmund´s Ham.'}
{'label': 'B', 'type': 'distractor_negation', 'text': 'Mluvčí nikdy ze Seaxmund´s Ham nepocházel.'}
{'label': 'C', 'type': 'distractor_lexical', 'text': 'Mluvčí kdysi v Seaxmund´s Ham přenocoval při cestě na jih.'}
{'label': 'D', 'type': 'distractor_associative', 'text': 'Seaxmund´s Ham je v této oblasti největším obchodním centrem.'}


In [25]:
# try conversion
converted_example= convert_item(data[6])
print(json.dumps(converted_example, ensure_ascii=False, indent=2))

{
  "id": "presupposition-factive-semifactive-7",
  "creation_method": "corpus_mining",
  "phenomenon": "presupposition",
  "category": "factive-semifactive",
  "context": "V emotivní hádce o polibek muž reaguje s nadhledem a snaží se situaci odlehčit.",
  "utterance": "Muž: „Zapomněl jsem, že je ti teprve devatenáct, a proto nechápeš, že láska není k polibku bezpodmínečně nutná.“",
  "question": "Co je bráno jako samozřejmý předpoklad, aby výpověď dávala smysl?",
  "options": [
    {
      "label": "A",
      "type": "presupposition",
      "text": "Dotyčné osobě je teprve devatenáct."
    },
    {
      "label": "B",
      "type": "distractor_negation",
      "text": "Této dotyčné osobě nikdy devatenáct let nebylo."
    },
    {
      "label": "C",
      "type": "distractor_lexical",
      "text": "Této dotyčné osobě je teprve devatenáct minut do konce směny."
    },
    {
      "label": "D",
      "type": "distractor_associative",
      "text": "Mnoho mladých lidí v devatenácti lete

In [26]:
# apply the convesrion on the whole dataset
converted_data = [convert_item(item) for item in data]

print(f"original items: {len(data)}")
print(f"converted items: {len(converted_data)}")

original items: 60
converted items: 60


In [27]:
# smth like a sanity check
for i, item in enumerate(converted_data):
    labels = [opt["label"] for opt in item["options"]]
    types = [opt["type"] for opt in item["options"]]

    if len(labels) != len(set(labels)):
        print(f"duplicate labels in item {i}: {item['id']}")

    if item["correct_option"] not in labels:
        print(f"correct_option missing among labels in item {i}: {item['id']}")

    if types.count("presupposition") != 1:
        print(f"expected exactly one 'presupposition' option in item {i}: {item['id']}")

print("sanity checks done")

sanity checks done


In [28]:
# save the converted dataset

from pathlib import Path
import json

path_output = Path(r"data\presupposition\eval")

out_path = path_output / file_name

with out_path.open("w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=2)

print(f"saved to: {out_path.resolve()}")

saved to: C:\Users\Admin\Desktop\hospoda.CZECH-PUB\code\czech-pub-copy-to-load\czech-pub\data\presupposition\eval\presupposition-factive-semifactive-60.json
